# Task 4: Preprocessing


Create the fixed Task 4 split and persist preprocessing configuration for the model notebooks.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import json

import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

from src.task4.config import (
    CONFIG_DIR,
    DATA_PATH,
    LABEL_COLUMN,
    LABEL_ID_COLUMN,
    SPLIT_DIR,
)
from src.task4.preprocessing import compute_rgb_mean_std

In [ ]:
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(DATA_PATH)
df.info()

In [ ]:
stratify_cols = [
    "articleType",
    "gender",
]

stratify_features = pd.get_dummies(
    df[stratify_cols].astype(str),
    prefix=stratify_cols,
)

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42,
)

outer_train_idx, test_idx = next(splitter.split(df, stratify_features))
outer_train_df = df.iloc[outer_train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train:", outer_train_df.shape)
print("Test: ", test_df.shape)

In [ ]:
for split_df in (outer_train_df, test_df):
    split_df[LABEL_COLUMN] = (
        split_df["articleType"].str.strip() + "__" + split_df["gender"].str.strip()
    )

classes = sorted(outer_train_df[LABEL_COLUMN].unique().tolist())
label_to_index = {label: index for index, label in enumerate(classes)}
outer_train_df[LABEL_ID_COLUMN] = outer_train_df[LABEL_COLUMN].map(label_to_index).astype("int64")
test_df[LABEL_ID_COLUMN] = test_df[LABEL_COLUMN].map(label_to_index).astype("Int64")

unseen_test_labels = sorted(set(test_df[LABEL_COLUMN]) - set(label_to_index))
if unseen_test_labels:
    print(
        f"Warning: {len(unseen_test_labels)} test label(s) are absent from training and have <NA> IDs."
    )
    print(unseen_test_labels)

with (CONFIG_DIR / "articleType_gender_label_encoder.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "label_column": LABEL_COLUMN,
            "label_id_column": LABEL_ID_COLUMN,
            "fit_split": "train",
            "separator": "__",
            "classes": classes,
            "label_to_index": label_to_index,
        },
        file,
        indent=2,
        sort_keys=True,
    )

outer_train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)

class_counts = outer_train_df[LABEL_COLUMN].value_counts()
print(f"Saved {len(classes)} classes")
print(f"Classes with fewer than 2 training examples: {(class_counts < 2).sum()}")


In [ ]:
train_mean, train_std = compute_rgb_mean_std(outer_train_df["id"])
print("Training RGB mean:", np.round(train_mean, 4))
print("Training RGB std: ", np.round(train_std, 4))

image_preprocessing_config = {
    "input_color_mode": "RGB",
    "resize": {
        "method": "letterbox",
        "target_size": [128, 128],
        "interpolation": "bilinear",
        "padding_color_rgb": [255, 255, 255],
        "centering": [0.5, 0.5],
    },
    "tensor": {
        "layout": "CHW",
        "dtype": "float32",
        "value_range_before_normalization": [0.0, 1.0],
    },
    "normalization": {
        "mean_rgb": train_mean,
        "std_rgb": train_std,
    },
}

with (CONFIG_DIR / "image_preprocessing.json").open("w", encoding="utf-8") as file:
    json.dump(image_preprocessing_config, file, indent=2)